In [82]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [83]:
df=pd.read_csv("../../data/raw/major_leagues v1/sofascore_major_leagues_2324season.csv")
df_yearly=pd.read_csv("../../data/raw/major_leagues v1/sofascore_MLS_Argentina_24season.csv")

In [84]:
df.shape

(8016, 117)

In [85]:
df_yearly.shape

(1733, 117)

In [86]:
set(df.columns) == set(df_yearly.columns)

True

In [87]:
df.columns.tolist() == df_yearly.columns.tolist()

True

In [88]:
total_df = pd.concat([df, df_yearly], ignore_index=True)

In [89]:
total_df.shape

(9749, 117)

In [90]:
total_df = total_df.rename(columns={col: f"{col}".lower() for col in total_df.columns})

In [91]:
total_df=total_df.drop(columns=["outfielderblocks","totwappearances","goalsprevented"])

In [92]:
total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]

C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\2912888129.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\2912888129.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]


In [93]:
per90_cols = [
    # Finishing
    "goals",
    "expectedgoals",
    "goals_minus_xg",
    "totalshots",
    "shotsontarget",
    "shotsofftarget",
    "shotsfrominsidethebox",
    "shotsfromoutsidethebox",
    "hitwoodwork",
    "leftfootgoals",
    "rightfootgoals",
    "headedgoals",
    "goalsfrominsidethebox",
    "goalsfromoutsidethebox",
    "freekickgoal",
    "penaltiestaken",
    "penaltygoals",
    "bigchancesmissed",

    # Creativity
    "assists",
    "expectedassists",
    "assists_minus_xa",
    "goalsassistssum",
    "totalpasses",
    "accuratepasses",
    "inaccuratepasses",
    "totaloppositionhalfpasses",
    "accurateoppositionhalfpasses",
    "totalownhalfpasses",
    "accurateownhalfpasses",
    "accuratefinalthirdpasses",
    "keypasses",
    "totalattemptassist",
    "passtoassist",
    "bigchancescreated",
    "totallongballs",
    "accuratelongballs",
    "totalchippedpasses",
    "accuratechippedpasses",
    "totalcross",
    "accuratecrosses",

    # Possession
    "touches",
    "possessionlost",
    "dispossessed",
    "totalcontest",
    "successfuldribbles",
    "offsides",
    "wasfouled",
    "fouls",

    # Defensive
    "tackles",
    "tackleswon",
    "interceptions",
    "ballrecovery",
    "possessionwonattthird",
    "clearances",
    "blockedshots",
    "dribbledpast",
    "errorleadtoshot",
    "errorleadtogoal",

    # Duels
    "totalduelswon",
    "duellost",
    "groundduelswon",
    "aerialduelswon",
    "aeriallost",
    
    # Goalkeeping
    "saves",
    "savescaught",
    "savesparried",
    "savedshotsfrominsidethebox",
    "savedshotsfromoutsidethebox",
    "runsout",
    "successfulrunsout",
    "goalkicks",
    "punches",
    "highclaims",
    "crossesnotclaimed",  
    

    # Discipline
    "yellowcards",
    "yellowredcards",
    "directredcards",
    "owngoals"
]

rate_cols = [
    "goalconversionpercentage",
    "scoringfrequency",

    "accuratepassespercentage",
    "accuratelongballspercentage",
    "successfuldribblespercentage",

    "tackleswonpercentage",

    "totalduelswonpercentage",
    "groundduelswonpercentage",
    "aerialduelswonpercentage",

    "penaltyconversion",

    "rating",
    "totalrating",
    "countrating"
]

for col in per90_cols:
    total_df[f"{col}_per90"] = np.where(
        total_df["minutesplayed"] > 0,
        (total_df[col] / total_df["minutesplayed"]) * 90,
        np.nan
    )

total_df["goals_per_xg"] = np.where(
    total_df["expectedgoals"] > 0,
    total_df["goals"] / total_df["expectedgoals"],
    1
)

total_df["shots_on_target_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsontarget"] / total_df["totalshots"],
    0
)


total_df["inside_box_shot_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsfrominsidethebox"] / total_df["totalshots"],
    0
)

total_df["assist_conversion"] = np.where(
    total_df["keypasses"] > 0,
    total_df["assists"] / total_df["keypasses"],
    0
)


total_df["xa_per_keypass"] = np.where(
    total_df["keypasses"] > 0,
    total_df["expectedassists"] / total_df["keypasses"],
    0
)


total_df["final_third_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accuratefinalthirdpasses"] / total_df["accuratepasses"],
    0
)

total_df["opp_half_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accurateoppositionhalfpasses"] / total_df["accuratepasses"],
    0
)

total_df["dribbles_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["successfuldribbles"] / total_df["touches"],
    0
)

total_df["dispossessed_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["dispossessed"] / total_df["touches"],
    0
)

total_df["possession_lost_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["possessionlost"] / total_df["touches"],
    0
)

total_df["defensive_actions"] = (
    total_df["tackles"] +
    total_df["interceptions"] +
    total_df["ballrecovery"]
)

total_df["defensive_actions_per90"] = (
    total_df["tackles_per90"] +
    total_df["interceptions_per90"] +
    total_df["ballrecovery_per90"]
)

C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\3909454853.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_per90"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\3909454853.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_per_xg"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\3909454853.py:130: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

In [94]:
total_df=total_df[total_df['minutesplayed'] > 900]

In [95]:
total_df['league'].value_counts()

league
USA MLS                       462
England EFL Championship      438
Spain La Liga 2               399
Argentina Liga Profesional    370
Italy Serie A                 345
Turkiye Super Lig             342
Spain La Liga                 341
Italy Serie B                 339
England Premier League        338
France Ligue 2                334
Germany Bundesliga            296
France Ligue 1                292
Germany 2.Bundesliga          278
Saudi Arabia Pro League       277
Portugal Primeira Liga        273
Netherlands Eredivisie        266
Name: count, dtype: int64

In [96]:
total_df.shape

(5390, 206)

In [97]:
metadata_features = [
    "player",
    "team",
    "player id",
    "team id",
    "league",
    "position",
    "minutesplayed",
    "matchesstarted",
    "appearances"
]

In [98]:
total_df=total_df.fillna(0.0)

In [99]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    null_values=total_df.isna().sum().sum()
    print(null_values)

0


In [100]:
total_rows = len(total_df)
unique_ids = total_df['player id'].nunique()
duplicate_count = total_rows - unique_ids
duplicate_count

25

In [101]:
features_to_scale = [
    col for col in total_df.columns 
    if total_df[col].dtype in [np.float64, np.int64] 
    and col not in metadata_features
]

for col in features_to_scale:
    grouped = total_df.groupby(["league", "position"])[col]
    group_mean = grouped.transform("mean")
    group_std = grouped.transform("std")
    
    total_df[f"{col}_zscore"] = np.where(
        group_std > 0,
        (total_df[col] - group_mean) / group_std,
        0.0
    )

C:\Users\vibha\AppData\Local\Temp\ipykernel_5444\1788466843.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_zscore"] = np.where(


In [102]:
total_df.columns.to_list()

['accuratechippedpasses',
 'accuratecrosses',
 'accuratecrossespercentage',
 'accuratefinalthirdpasses',
 'accuratelongballs',
 'accuratelongballspercentage',
 'accurateoppositionhalfpasses',
 'accurateownhalfpasses',
 'accuratepasses',
 'accuratepassespercentage',
 'aerialduelswon',
 'aerialduelswonpercentage',
 'aeriallost',
 'appearances',
 'assists',
 'attemptpenaltymiss',
 'attemptpenaltypost',
 'attemptpenaltytarget',
 'ballrecovery',
 'bigchancescreated',
 'bigchancesmissed',
 'blockedshots',
 'cleansheet',
 'clearances',
 'countrating',
 'crossesnotclaimed',
 'directredcards',
 'dispossessed',
 'dribbledpast',
 'duellost',
 'errorleadtogoal',
 'errorleadtoshot',
 'expectedassists',
 'expectedgoals',
 'fouls',
 'freekickgoal',
 'goalconversionpercentage',
 'goalkicks',
 'goals',
 'goalsassistssum',
 'goalsconceded',
 'goalsconcededinsidethebox',
 'goalsconcededoutsidethebox',
 'goalsfrominsidethebox',
 'goalsfromoutsidethebox',
 'groundduelswon',
 'groundduelswonpercentage',
 'h

In [103]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[total_df.duplicated(subset=['player id'], keep=False)].drop_duplicates(subset=['player id'])[['player id', 'player']])

,player id,player
34,78152,Matz Sels
134,118179,Tim Ream
163,1117223,Maxime Estève
188,1009334,Lorenz Assignon
275,1109771,Adam Wharton
880,1084399,Carlos Vicente
1832,295133,Edoardo Goldaniga
1895,324353,Aleksey Miranchuk
2126,1069244,Tijjani Noslin
2146,877201,Gabriel Strefezza


In [104]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[['team', 'team id']].drop_duplicates().dropna())

,team,team id
2,West Ham United,37
3,Manchester United,35
4,Everton,48
5,Newcastle United,39
7,Brighton & Hove Albion,30
8,Wolverhampton,3
9,Chelsea,38
10,Bournemouth,60
11,Aston Villa,40
13,Fulham,43


In [105]:
player_team_ids = {
    78152: 14,        # Matz Sels -> Nottingham Forest
    118179: 43,       # Tim Ream -> Fulham
    1117223: 6,       # Maxime Estève -> Burnley
    1009334: 6,       # Lorenz Assignon -> Burnley
    1109771: 7,       # Adam Wharton -> Crystal Palace
    1084399: 2885,    # Carlos Vicente -> Deportivo Alavés
    295133: 2704,     # Edoardo Goldaniga -> Como
    324353: 2686,     # Aleksey Miranchuk -> Atalanta
    1069244: 2701,    # Tijjani Noslin -> Hellas Verona
    877201: 2689,     # Gabriel Strefezza -> Lecce
    1402217: 1656,    # Panos Katseris -> Lorient
    96116: 9,         # Kevin Long -> Birmingham City
    920936: 36,       # Lewis O'Brien -> Middlesbrough
    826155: 96,       # Abdülkadir Ömür -> Hull City
    140479: 36,       # Matt Crooks -> Middlesbrough
    1123234: 3035,    # Ronald -> CF Estrela da Amadora
    556884: 2846,     # Matías Dituro -> Elche
    174943: 2852,     # Carlos Izquierdoz -> Sporting Gijón
    886737: 3061,     # Derrick Köhn -> Galatasaray
    1084967: 2761,    # Gonzalo Abrego -> Cremonese
    182951: 274650,   # Maxime Chanot -> LAFC
    335747: 243211,   # Stian Gregersen -> Atlanta United
    830293: 5136,     # Bruno Wilson -> Vizela
    927038: 243211,   # Pedro Amador -> Atlanta United
    31981: 34313      # Éver Banega -> Al-Shabab
}

In [106]:
for player_id, target_team_id in player_team_ids.items():
    player_rows = total_df[total_df["player id"] == player_id]
    if len(player_rows) <= 1:
        continue
        
    total_mins = player_rows["minutesplayed"].sum()
    if total_mins == 0:
        continue
        
    target_idx = player_rows[player_rows["team id"] == target_team_id].index
    if len(target_idx) == 0:
        target_idx = [player_rows.index[0]]
        
    keep_idx = target_idx[0]
    drop_indices = player_rows.index[player_rows.index != keep_idx]
    
    total_df.loc[keep_idx, "minutesplayed"] = total_mins
    
    for col in total_df.columns:
        if col in metadata_features and col != "minutesplayed":
            continue
            
        if col.endswith("_zscore"):
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col.endswith("_per90") or col in per90_cols:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col in rate_cols or col in ["goals_per_xg", "shots_on_target_pct", "inside_box_shot_pct", "assist_conversion", "xa_per_keypass", "final_third_pass_pct", "opp_half_pass_pct", "dribbles_per_touch", "dispossessed_per_touch", "possession_lost_per_touch", "weak_foot_goals_pct"]:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
            
    total_df = total_df.drop(drop_indices)

total_df = total_df.reset_index(drop=True)

In [107]:
total_df.to_csv("../../data/processed/major_leagues/Sofascore_player_data_2324.csv",index=False)